# Setup

## Imports

In [ ]:
import sys, os
sys.path.insert(0, 'src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch, math
import random
import re
from collections import Counter
from rank_bm25 import BM25Okapi
import torch

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from data import (download_msmarco, download_squad, download_nq,
                  chunk_pdfs, load_jsonl, TripletDataset)
from model import TextEncoder
from losses import InfoNCELoss
from retrieval import DenseRetriever, BM25Retriever, TFIDFRetriever
from metrics import run_evaluation, print_metrics
from train import train, load_model, MODEL_NAME
from tokenizer import build_bpe_tokenizer, load_tokenizer

random.seed(42)

In [ ]:
data_path = 'data/processed'
data_raw = 'data/raw'
report_path = 'report'

wiki_chunks_path = f'{data_path}/chunks_wiki.jsonl'
nlp_chunks_path = f'{data_path}/chunks_nlp.jsonl'
oov_chunks_path = f'{data_path}/chunks_oov.jsonl'

wiki_eval_path = f'{data_path}/eval_wiki.jsonl'
nlp_eval_path = f'{data_path}/eval_nlp.jsonl'
oov_eval_path = f'{data_path}/eval_oov.jsonl'

## Data Loading

In [ ]:
download_msmarco(train_size=100000, val_size=5000, out_dir=data_path)

In [ ]:
download_squad(train_size=80000, val_size=5000, out_dir=data_path)

In [ ]:
download_nq(train_size=200000, val_size=5000, out_dir=data_path)

# Data Analysis

In [ ]:
ds = 'msmarco'
# ds = 'squad'
# ds = 'nq'
# ds = 'book'

train_path = f'{data_path}/{ds}_train_triplets.jsonl'
val_path = f'{data_path}/{ds}_val_triplets.jsonl'

In [ ]:
train_triplets = []
with open(train_path) as f:
    for line in f:
        train_triplets.append(json.loads(line))

print(f'Train triplets: {len(train_triplets):,}')
print()
print('Sample triplet:')
t = train_triplets[0]
print(f'Query : {t["query"]}')
print(f'Pos   : {t["pos"]}')
print(f'Neg   : {t["neg"]}')

In [ ]:
q_lens = [len(t['query'].split()) for t in train_triplets]
p_lens = [len(t['pos'].split()) for t in train_triplets]
n_lens = [len(t['neg'].split()) for t in train_triplets]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
graph_names = ['Query length (words)', 'Positive length (words)', 'Negative length (words)']
for ax, lens, label in zip(axes, [q_lens, p_lens, n_lens], graph_names):
    ax.hist(lens, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(label)
    ax.set_xlabel('Words')
    ax.axvline(np.mean(lens), color='red', linestyle='--', label=f'mean={np.mean(lens):.1f}')
    ax.legend()
plt.suptitle(f'{ds.upper()} Training Data Length Distributions', fontsize=13)
plt.tight_layout()
plt.savefig(f'{report_path}/{ds}_data_length_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
sample = random.sample(train_triplets, min(500, len(train_triplets)))

pos_scores, neg_scores = [], []
for t in sample:
    q_toks = t['query'].lower().split()
    corpus = [t['pos'].lower().split(), t['neg'].lower().split()]
    bm = BM25Okapi(corpus)
    sc = bm.get_scores(q_toks)
    pos_scores.append(sc[0])
    neg_scores.append(sc[1])

gaps = [p - n for p, n in zip(pos_scores, neg_scores)]
pct_pos_higher = sum(1 for g in gaps if g > 0) / len(gaps)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(gaps, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', label='gap=0')
axes[0].set_title('Negative Hardness: BM25 Score Gap (pos - neg)')
axes[0].set_xlabel('Score difference')
axes[0].legend()

lim = max(max(pos_scores), max(neg_scores)) * 1.05
axes[1].scatter(pos_scores, neg_scores, alpha=0.3, s=10)
axes[1].plot([0, lim], [0, lim], 'r--', label='equal')
axes[1].set_xlabel('Positive BM25 score'); axes[1].set_ylabel('Negative BM25 score')
axes[1].set_title('Pos vs Neg BM25 Scores')
axes[1].legend()

print(f"Positive ranked higher by BM25 : {pct_pos_higher:.1%}")
print(f"Mean score - pos: {sum(pos_scores)/len(pos_scores):.2f}  neg: {sum(neg_scores)/len(neg_scores):.2f}")

plt.tight_layout()
plt.savefig(f'{report_path}/{ds}_triplet_quality.png', bbox_inches='tight')
plt.show()

In [ ]:
def _tok_split(text):
    return re.sub(r"[^a-z0-9\s'\-]", " ", text.lower()).split()

all_sources = [
    (f'{data_path}/msmarco_train_triplets.jsonl', ['query', 'pos', 'neg']),
    (f'{data_path}/squad_train_triplets.jsonl',   ['query', 'pos', 'neg']),
    (f'{data_path}/book_train_triplets.jsonl', ['query', 'pos', 'neg']),
]

freq = Counter()
total_tokens = 0

for path, fields in all_sources:
    try:
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                for field in fields:
                    if r.get(field):
                        words = _tok_split(r[field])
                        freq.update(words)
                        total_tokens += len(words)
        print(f"Loaded {path}")
    except FileNotFoundError:
        print(f"Skipped {path}")

jm = load_jsonl(nlp_chunks_path)
for c in jm:
    words = _tok_split(c['text'])
    freq.update(words)
    total_tokens += len(words)
print(f"Loaded J&M chunks ({len(jm)} chunks)")

print()
print(f"Total tokens : {total_tokens:,}")
print(f"Unique words : {len(freq):,}")

print()
print(f"{'min_freq':>8}  {'vocab_size':>12}  {'coverage':>10}")
for mf in [1, 2, 5, 10, 20, 50, 100]:
    v = {w for w, c in freq.items() if c >= mf}
    cov = sum(c for w, c in freq.items() if w in v) / total_tokens
    print(f"  {mf:>8}  {len(v):>12,}  {cov:>9.2%}")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

sorted_freqs = sorted(freq.values(), reverse=True)
axes[0].loglog(range(1, len(sorted_freqs) + 1), sorted_freqs, lw=1, color='steelblue')
axes[0].set_xlabel('Word rank (log)'); axes[0].set_ylabel('Frequency (log)')
axes[0].set_title('Word frequency (Zipf law)')
axes[0].grid(True, which='both', alpha=0.3)

cum, cx, cy = 0, [], []
for rank, f in enumerate(sorted_freqs, 1):
    cum += f
    cx.append(rank); cy.append(cum / total_tokens)
axes[1].plot(cx, cy, lw=1.5, color='steelblue')
for target, color in [(0.90, 'orange'), (0.95, 'red'), (0.99, 'purple')]:
    for rank, cov in zip(cx, cy):
        if cov >= target:
            axes[1].axvline(rank, linestyle='--', color=color, alpha=0.7,
                            label=f'{target:.0%} @ {rank:,}')
            break
axes[1].set_xlabel('Vocabulary size'); axes[1].set_ylabel('Token coverage')
axes[1].set_title('Coverage curve'); axes[1].legend(fontsize=8)
axes[1].set_xlim(0, min(200_000, len(sorted_freqs))); axes[1].grid(True, alpha=0.3)

cutoffs = [1, 2, 5, 10, 20, 50, 100]
vsizes  = [sum(1 for c in freq.values() if c >= mf) for mf in cutoffs]
axes[2].bar(range(len(cutoffs)), vsizes, tick_label=cutoffs, color='steelblue', edgecolor='white')
axes[2].set_xlabel('min_freq'); axes[2].set_ylabel('Unique words')
axes[2].set_title('Vocab size vs. min_freq cutoff'); axes[2].set_yscale('log')
for i, v in enumerate(vsizes):
    axes[2].text(i, v * 1.15, f'{v:,}', ha='center', va='bottom', fontsize=7)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{report_path}/word_frequency_analysis.png', bbox_inches='tight')
plt.show()


In [ ]:
# Run after building the BPE tokenizer in the Training section
bpe_tok = load_tokenizer(f'{data_path}/bpe_tokenizer.json')
chunks = load_jsonl(nlp_chunks_path)

sample_chunks = chunks[:100]
words_total = sum(len(c['text'].split()) for c in sample_chunks)
subwords_total = sum(len(bpe_tok.encode(c['text'])) for c in sample_chunks)

print(f"BPE vocab size : {len(bpe_tok):,}")
print(f"Sample words : {words_total:,}")
print(f"Subword tokens : {subwords_total:,}")

history_path = 'checkpoints/TextEncoder_epoch_10.pt'
print(f"Avg tokens/word : {subwords_total/words_total:.2f}  (1.0 = no splitting)")

In [ ]:
samples = random.sample(train_triplets, 5)

for i, t in enumerate(samples, 1):
    print(f"Query : {t['query']}")
    print(f"Positive : {t['pos']}")
    print(f"Negative : {t['neg']}")
    print()

## Chunking

In [ ]:
# Wiki entry
wiki_chunks = chunk_pdfs(f'{data_raw}/ww2.pdf', out_path=wiki_chunks_path)
print()
# NLP book
nlp_chunks = chunk_pdfs(f'{data_raw}/jm.pdf', out_path=nlp_chunks_path)
print()
# OOV entry
oov_chunks = chunk_pdfs(f'{data_raw}/computation.pdf', out_path=oov_chunks_path)

In [ ]:
chunks = load_jsonl(nlp_chunks_path)
print(f'Total J&M chunks: {len(chunks)}')

chunk_lens = [c['word_count'] for c in chunks]
print(f'Word count: min={min(chunk_lens)}, max={max(chunk_lens)}, mean={np.mean(chunk_lens):.1f}')

print('Sample chunks:')
for c in chunks[1000:1002]:
    print(f"{c['id']} : {c['text']}")

# Baseline Evaluation

In [ ]:
# base_chunks_path = wiki_chunks_path
# base_eval_path = wiki_eval_path

base_chunks_path = nlp_chunks_path
base_eval_path = nlp_eval_path

# base_chunks_path = oov_chunks_path
# base_eval_path = oov_eval_path

In [ ]:
chunks = load_jsonl(base_chunks_path)

bm25_ret = BM25Retriever()
bm25_ret.build(chunks)
bm25_ret.save()

tfidf_ret = TFIDFRetriever()
tfidf_ret.build(chunks)
tfidf_ret.save()

In [ ]:
# Sanity check
# test_q = 'What is the Viterbi algorithm?'
test_q = 'What is the difference between precision and recall?'
print('BM25 top-5:')
for score, text, chunk_idx in bm25_ret.search(test_q, k=5):
    print(f'{chunk_idx} [{score:.2f}] : {text}')

print('\nTF-IDF top-5:')
for score, text, chunk_idx in tfidf_ret.search(test_q, k=5):
    print(f'{chunk_idx} [{score:.4f}] : {text}')

In [ ]:
eval = load_jsonl(base_eval_path)
print(f'Eval queries: {len(eval)}')

bm25_metrics  = run_evaluation(bm25_ret,  eval)
tfidf_metrics = run_evaluation(tfidf_ret, eval)

print_metrics(bm25_metrics,  name='BM25')
print_metrics(tfidf_metrics, name='TF-IDF')

# Training

## Vocab creation

In [ ]:
# Default Model
corpus_paths = [
    f'{data_path}/msmarco_train_triplets.jsonl',
    # f'{data_path}/squad_train_triplets_old.jsonl',
    f'{data_path}/book_train_triplets.jsonl',
    # f'{data_path}/nq_train_triplets.jsonl',
]
chunk_texts = [c['text'] for c in load_jsonl(nlp_chunks_path)]

tokenizer = build_bpe_tokenizer(
    corpus_paths=corpus_paths,
    extra_texts=chunk_texts,
    save_dir=data_path,
    vocab_size=50000,
)
print(f'Vocab size: {len(tokenizer):,}')

In [ ]:
_tok = load_tokenizer("data/processed/vocab.json")

def _split(text):
    return re.sub(r"[^a-z0-9\s'\-]", " ", text.lower()).split()

oov_counter = Counter()
total, oov_total = 0, 0

for c in load_jsonl(nlp_chunks_path):
    for tok in _split(c["text"]):
        total += 1
        if tok not in _tok.word2id:
            oov_counter[tok] += 1
            oov_total += 1

print(f"Vocab size : {_tok.vocab_size:,}")
print(f"Total tokens in J&M : {total:,}")
print(f"OOV tokens : {oov_total:,}  ({oov_total/total:.1%})")
print()
print("Top-50 most frequent OOV words in J&M chunks:")
print(f"{'word':<20} {'count':>6}")
for word, count in oov_counter.most_common(50):
    print(f"{word:<20} {count:>6}")

In [ ]:
model = TextEncoder()
print(f'TextEncoder parameters: {model.count_parameters():,}')

In [ ]:
dummy_ids  = torch.randint(1, 1000, (4, 64))
dummy_mask = torch.ones(4, 64, dtype=torch.long)
out = model(dummy_ids, dummy_mask)
print(f'Output shape: {out.shape}') # (4, 256)
print(f'L2 norms: {out.norm(dim=-1)}') # all should be about 1

In [ ]:
criterion = InfoNCELoss(temperature=1.0)
B, D = 64, 256
q = torch.nn.functional.normalize(torch.randn(B, D), dim=-1)
p = torch.nn.functional.normalize(torch.randn(B, D), dim=-1)
loss = criterion(q, p)
expected = math.log(B)
print(f'InfoNCE loss (random): {loss.item():.4f} (expected ~ {expected:.4f})')

In [ ]:
train()

In [ ]:
history_path = f'checkpoints/{MODEL_NAME}/training_history.json'
with open(history_path) as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

steps  = history['train_step']
losses = history['train_loss']
axes[0].plot(steps, losses, alpha=0.2, color='steelblue')
window = min(100, len(losses))
if len(losses) >= window:
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    axes[0].plot(steps[window-1:], smoothed, color='steelblue', label=f'{window}-step avg')
    axes[0].legend()
axes[0].set_title('Train Loss'); axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')

axes[1].plot(history['val_epoch'], history['val_loss'], marker='o', color='coral')
axes[1].set_title('Val Loss per Epoch'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')

axes[2].plot(steps, history['train_lr'], color='green')
axes[2].set_title('Learning Rate Schedule'); axes[2].set_xlabel('Step'); axes[2].set_ylabel('LR')

plt.suptitle('TextEncoder Training Curves', fontsize=13)
plt.tight_layout()
os.makedirs(report_path, exist_ok=True)
plt.savefig(f'{report_path}/{MODEL_NAME}_training_curves.png', bbox_inches='tight')
plt.show()
print(f"Saved to {report_path}/{MODEL_NAME}_training_curves.png")

# Results & Comparison

In [ ]:
ds_name = 'nlp' # 'wiki' 'nlp' 'oov'

# res_chunks_path = wiki_chunks_path
# res_eval_path = wiki_eval_path

res_chunks_path = nlp_chunks_path
res_eval_path = nlp_eval_path

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer = load_model(f'checkpoints/{MODEL_NAME}/best_model.pt', device=device)

chunks = load_jsonl(res_chunks_path)

dense_ret = DenseRetriever(d_model=256)
dense_ret.build(chunks, model, tokenizer, device=device)
dense_ret.save()

bm25_ret = BM25Retriever()
bm25_ret.build(chunks)
bm25_ret.save()

tfidf_ret = TFIDFRetriever()
tfidf_ret.build(chunks)
tfidf_ret.save()

In [ ]:
test_text = chunks[0]['text']
results = dense_ret.search(test_text, model, tokenizer, k=1, device=device)
print(f'Self-retrieval score: {results[0][0]:.6f}  (expected ≈ 1.0)')

In [ ]:
eval_qrels = load_jsonl(res_eval_path)

bm25_ret.load('indexes/bm25.pkl')
tfidf_ret.load('indexes/tfidf.pkl')

bm25_metrics   = run_evaluation(bm25_ret,  eval_qrels)
tfidf_metrics  = run_evaluation(tfidf_ret, eval_qrels)
neural_metrics = run_evaluation(
    dense_ret, eval_qrels,
    model=model, tokenizer=tokenizer, device=device
)

print_metrics(bm25_metrics, 'BM25')
print_metrics(tfidf_metrics, 'TF-IDF')
print_metrics(neural_metrics, 'Neural (TextEncoder + Dense)')

In [ ]:
all_metrics = {
    'BM25':   bm25_metrics,
    'TF-IDF': tfidf_metrics,
    'Neural': neural_metrics,
}
df = pd.DataFrame(all_metrics).T
df = df[sorted(df.columns)]
print(df.to_string(float_format='{:.4f}'.format))

metrics_to_plot = ['Recall@1', 'Recall@5', 'Recall@10', 'MRR@10', 'NDCG@10']
plot_df = df[metrics_to_plot]
ax = plot_df.plot(kind='bar', figsize=(12, 5), rot=0)
ax.set_title('Retriever Comparison')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'report/{MODEL_NAME}_{ds_name}_metric_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
queries = [q['query'] for q in eval_qrels[:5]]
for query in queries:
    print(f'\nQuery: {query}')
    print('BM25 top-1 :', bm25_ret.search(query, k=1)[0][1])
    print('TF-IDF top-1 :', tfidf_ret.search(query, k=1)[0][1])
    print('Neural top-1 :', dense_ret.search(query, model, tokenizer, k=1, device=device)[0][1])